# PHOEBE vs SPICE — paper figures

Interactive front end to `make_paper_figures.py`. The plotting code is *imported*,
not copied, so this notebook and the script can never drift apart — but the
palette, style and per-figure parameters are all overridable from here.

**Two levels of tinkering**

| you want to change | where |
|---|---|
| colours, dashes, line widths, fonts, figure size | the **Style** cell below — re-run it, then re-run any figure cell |
| the plot itself (panel layout, what is drawn) | edit `make_paper_figures.py`; `autoreload` picks it up, no kernel restart |

Results come from `atmosphere_out/<suite>/`, written by `spice_vs_phoebe_atmospheres.py`.
Nothing here recomputes any physics — it only reads the pickles.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, pickle, sys
os.environ.setdefault("JAX_PLATFORMS", "cpu")   # figures need no GPU
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

HERE = Path('/scratch/y89/mj8805/spice/tutorial/paper_results/phoebe_spice_comparison/')
sys.path.insert(0, str(HERE))
import make_paper_figures as mpf

RESULTS = HERE / "atmosphere_out"
FIGDIR  = HERE / "atmosphere_out" / "notebook_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
print("results :", RESULTS)
print("suites  :", [d.name for d in sorted(RESULTS.iterdir()) if d.is_dir()])

## Style — edit here

`ATM_COLOR` is keyed by **atmosphere**, not by code: the same model is the same
colour in every panel, and SPICE-vs-PHOEBE is carried by linestyle instead. That
way identity never depends on colour alone, which keeps the figures readable in
greyscale and for colour-blind readers.

The default palette is Okabe–Ito-derived and passes all six checks of the dataviz
validator against a light surface (lightness band, chroma floor, CVD separation,
normal-vision floor, contrast ≥ 3:1). **If you change these, re-check CVD
separation** — the pair most at risk is whichever two hues end up adjacent in a
legend.

`ATM_DASH` matters more than it looks: SPICE and ck2004 agree so closely that a
thin line disappears under the other. SPICE is drawn as a thick translucent band
with the PHOEBE dashes riding *inside* it, so agreement reads as agreement rather
than as a missing series.

In [ ]:
# --- colours: keyed by atmosphere -------------------------------------------
mpf.ATM_COLOR.update({
    "aemu":            "#0072B2",   # SPICE MARCS emulator   (blue)
    "spice_blackbody": "#0072B2",   # SPICE Blackbody        (same identity)
    "ck2004":          "#D55E00",   # PHOEBE ck2004          (vermillion)
    "phoenix":         "#009E73",   # PHOEBE phoenix         (bluish green)
    "blackbody":       "#8C6D1F",   # blackbody              (ochre)
})

# --- dash patterns: one per PHOEBE atmosphere -------------------------------
mpf.ATM_DASH.update({
    "ck2004":    (5.0, 1.6),
    "phoenix":   (1.8, 1.6),
    "blackbody": (7.0, 2.2),
})

# --- the SPICE band underneath ----------------------------------------------
mpf.SPICE_LW    = 2.6      # width of the translucent SPICE line
mpf.SPICE_ALPHA = 0.55

# --- labels (legend text) ----------------------------------------------------
mpf.ATM_LABEL.update({
    "aemu":            "SPICE (MARCS emulator)",
    "spice_blackbody": "SPICE (Blackbody)",
    "ck2004":          "PHOEBE ck2004",
    "phoenix":         "PHOEBE phoenix",
    "blackbody":       "blackbody",
})

# --- global matplotlib style -------------------------------------------------
mpf.setup_style()          # boxed axes, no grid, paper font sizes
plt.rcParams.update({
    "font.size":       8,
    "figure.dpi":    120,   # on-screen only; savefig stays at 300
    "savefig.dpi":   300,
})

print("style applied — re-run any figure cell below")

## Load the results

Each suite is a separate comparison and must not be mixed:

* **`atmospheres`** — SPICE aemu vs PHOEBE ck2004 + phoenix. Both sides carry
  their own limb darkening, so the residual is radiative transfer.
* **`blackbody`** — SPICE Blackbody vs PHOEBE blackbody, identical physics on
  both sides. The control: whatever is left is geometry, meshing and passband
  integration, and it sets the floor the other suite is measured against.

`set_suite` fixes which atmospheres and which SPICE emulator every subsequent
figure uses, so a figure can never accidentally mix the two.

In [ ]:
def load(suite, what="binary", sub=None):
    """Read one results pickle. what: 'binary' | 'intensity'; sub e.g. 'dense', 'mesh5120'."""
    name = {"binary": "binary_light_curves.pkl", "intensity": "intensity_grid.pkl"}[what]
    path = RESULTS / suite / (sub or "") / name
    if not path.exists():
        print(f"missing: {path}")
        return None
    with open(path, "rb") as f:
        return pickle.load(f)

def use(suite):
    """Point the figure functions at one suite (sets atmospheres + SPICE side)."""
    grid   = load(suite, "intensity")
    binary = load(suite, "binary")
    atms = (grid or {}).get("phoebe_atms")
    if atms is None and binary:
        atms = next(iter(binary.values())).get("atms")
    kind = mpf.spice_key_of(binary) if binary else ("blackbody" if suite == "blackbody" else "aemu")
    mpf.set_suite(atms or mpf.PHOEBE_ATMS, kind)
    print(f"suite={suite}  SPICE={kind}  PHOEBE={mpf.PHOEBE_ATMS}")
    return grid, binary

SUITE = "atmospheres"          # <-- switch to "blackbody" for the control
grid, binary = use(SUITE)

## Display helper

The figure functions write files and close the figure (they are built for batch
use), so this displays the saved PNG inline. Setting `SAVE_FORMATS` to include
`"png"` means the notebook needs no PDF renderer — `pymupdf` is not present in
every kernel, and the paper still gets vector PDFs.

In [ ]:
# Figures are written as PDF (for the paper) and PNG (to display here).
mpf.SAVE_FORMATS = ("pdf", "png")

def show(names, width=900):
    """Display saved figure(s) inline. No PDF renderer needed."""
    from IPython.display import Image, display
    for name in ([names] if isinstance(names, str) else names):
        png = (FIGDIR / name).with_suffix(".png")
        if not png.exists():
            print(f"missing {png}"); continue
        print(name)
        display(Image(filename=str(png), width=width))

## Parameter-dependence figures

These come from the intensity grid — single-star passband quantities over
Teff × log g × [M/H], no orbit involved. They are what predict the eclipse
behaviour further down.

In [ ]:
show(mpf.fig_response_axes(grid, FIGDIR))    # Δlog I vs each axis, 4 bands × 3 axes

In [ ]:
show(mpf.fig_response_maps(grid, FIGDIR))    # the Teff–log g plane, values printed

In [ ]:
show(mpf.fig_parameter_sweep(grid, FIGDIR))  # median/max over the grid (log axis)

## Limb darkening

`ldint = 2∫μL(μ)dμ` is PHOEBE's convention — a uniform disc is exactly 1. It is
the factor converting normal intensity to flux, so a disagreement here scales
eclipse depth directly.

In [ ]:
show(mpf.fig_limb_darkening(grid, FIGDIR))   # I(μ)/I(1) + residuals
show(mpf.fig_ldint(grid, FIGDIR))           # ldint and linear u vs Teff

## Eclipse figures

In [ ]:
show([mpf.fig_eclipse_depths(binary, FIGDIR),
      mpf.fig_depth_residuals(binary, FIGDIR),
      mpf.fig_light_ratios(binary, FIGDIR)])

In [ ]:
show(mpf.fig_binary_lightcurves(binary, FIGDIR))
show(mpf.fig_eclipse_residual_stack(binary, FIGDIR))   # one per system

## Mesh convergence

Reads every `mesh<N>/` directory in the suite. In the **blackbody** suite the
physics is identical on both sides, so the residual is purely numerical and must
fall as the meshes refine — that fall calibrates what the atmospheres suite can
resolve. In the **atmospheres** suite it stays flat, because the difference is
physical.

In [ ]:
sweep = mpf.load_mesh_sweep(RESULTS, SUITE)
print("resolutions:", sorted(sweep))
if len(sweep) >= 2:
    show(mpf.fig_mesh_convergence(sweep, FIGDIR, SUITE))

## Eclipse shape (ingress / egress)

Depth is set by the **light ratio**; shape is set by **limb darkening**. Dividing
each curve by its own maximum depth removes the first and leaves the second, so
these are the sharpest atmosphere discriminant in the suite.

Needs the dense runs (`--n-per-eclipse 60`); with only a handful of points the
"shape" is an artefact of where the samples fell.

In [ ]:
dense = load(SUITE, "binary", sub="dense")
ctrl  = load("blackbody", "binary", sub="dense")   # drawn as the numerical floor
if dense:
    show(mpf.fig_ingress_shape(dense, FIGDIR))
    show(mpf.fig_shape_summary(dense, FIGDIR, ctrl))

## Regenerate everything, both suites

Equivalent to running `python make_paper_figures.py` from the shell — writes to
each suite's own `figures/` directory rather than the notebook one.

In [ ]:
# !python make_paper_figures.py